<p style="text-align: center">
<img src="../../assets/images/dtlogo.png" alt="Duckietown" width="50%">
</p>

# Évitement des obstacles

Enfin, nous utiliserons notre modèle de détection d'objets pour éviter les piétons canards lors de notre navigation dans Duckietown.

Le fichier que nous utiliserons pour cela est [`model.py`](../../packages/solution/model.py).

La fonction que nous devons écrire, avec quelques fonctions auxiliaires, est `get_wheel_velocities_from_image`. Vous pouvez vous attendre à ce que cette fonction soit appelée à chaque nouvelle image de la caméra, et nous devrions renvoyer des commandes pour les roues. Notre stratégie est très simple : si nous détectons un piéton Duckie avec un niveau de confiance suffisamment élevé et à une distance suffisamment proche, nous nous arrêterons (commandes de roue nulles) ; sinon, nous enverrons des valeurs de commande fixes pour que le robot avance en ligne droite (approximativement comme le véritable Duckiebot). 


La structure de cette fonction est la suivante :

```python
    def get_wheel_velocities_from_image(self, img: np.ndarray):
        try:
            detections = self._run_detector(img)
        except Exception as e:
            print(f"ONNX inference error {e}")
            return DifferentialPWM(left=0.0, right=0.0)
        if self._should_stop(detections):
            return DifferentialPWM(left=0.0, right=0.0)
        else:
            return DifferentialPWM(left=self.forward_pwm, right=self.forward_pwm)
```
Notez que `FORWARD_PWM` est défini dans votre fichier [solution/config.py](../../packages/solution/config.py). Cela déterminera la vitesse de déplacement de votre robot lorsqu'il n'y a (normalement) aucun canard sur son chemin.

Nous effectuons deux appels de fonction : le premier est `run_detector`.

Cette fonction appelle simplement votre modèle ONNX et renvoie les détections.

```python
    def _run_detector(self, img_bgr):
        x = self._preprocess(img_bgr)
        out = self.session.run(None, {self.input_name: x})[0]  # shape [1,N,6]
        return out[0]
```

Votre tâche consiste à traiter les détections et à décider s'il convient de poursuivre ou non le traitement. Pour chaque détection, l'arrêt doit être effectué si les deux conditions suivantes sont remplies :

1. Le score de confiance est supérieur au seuil `CONF_THRESHOLD` défini dans [solution/config.py](../../packages/solution/config.py).

2. Le canard est plus proche que la distance `STOP_DISTANCE`. Vous devez trouver un moyen d'estimer cette distance.

Le code ressemble à ceci :

```python
    def _should_stop(self, detections: np.ndarray): 
        
        stop = False

        for x1, y1, x2, y2, score, _ in detections:


        # TODO : nous ne voulons pas prendre en compte les détections dont le score de confiance est inférieur à CONF_THRESHOLD (une valeur à définir dans config.py).

        # TODO : nous voulons arrêter la détection si un canard se trouve à une distance inférieure à STOP_DISTANCE.

        # Pour calculer si le canard est trop proche, nous devons convertir les coordonnées de pixels en
        # coordonnées du monde. Pour ce faire, vous pouvez utiliser l'objet `self.ground_projector` qui a
        # chargé le calibration extrinsèque de la caméra.

        # Plus précisément, si vous souhaitez projeter un objet de type `pix = Pixel(x=u, y=v)` sur un point du plan du sol,

        # vous pouvez d'abord le convertir en vecteur (`vec = self.ground_projector.camera.pixel2vector(pix)`) et

        # ensuite, vous pouvez intersecter ce vecteur avec le plan du sol (`self.ground_projector.vector2ground(vec)`).

        # Ce sera le point sur le plan du sol correspondant au pixel d'entrée.

        return stop
```

Une fois votre implémentation satisfaisante, vous pouvez la tester, comme décrit dans le [`README`](../../README.md). 

Si tout se passe bien, votre Duckiebot devrait S'ARRÊTER avant que des canards ne soient blessés !

Il peut être utile de visualiser les détections effectuées par votre modèle. Vous pouvez le faire avec la visionneuse d'images. Ouvrez-la avec

```bash
dts duckiebot image_viewer ROBOT_NAME
```

Dans le menu déroulant en haut, vous devriez pouvoir sélectionner le nom de sujet `/ROBOT_NAME/object_detector_image/jpeg`. Vous verrez alors une image avec les cadres de délimitation superposés et leurs niveaux de confiance associés.

![image viewer object detection](../../assets/images/duckie_detection.png)
